# Copy Suppression

Analyzes heads that suppress the copying of tokens from context — the "negative heads" phenomenon. These heads attend to positions and push the residual stream *away* from predicting those tokens.

**References:**
- McDougall et al. (2023) "Copy Suppression: Comprehensively Understanding an Attention Head"
- Wang et al. (2022) "Interpretability in the Wild: GPT-2 IOI"

## Why This Matters

Copy suppression heads actively prevent the model from copying tokens that appear in the input but shouldn't be predicted. Understanding these 'negative' heads — which tokens they suppress and when — is essential for understanding tasks like IOI where the model must distinguish between repeated and non-repeated names.

**Key references:**
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.copy_suppression import (
    copy_suppression_score,
    find_negative_heads,
    suppression_per_attended_token,
    copy_vs_suppress_decomposition,
    suppression_circuit_on_ioi,
)

# Create a small model
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

## 1. Copy Suppression Score

For a given head, compute how much it suppresses the logit of each attended-to token. Positive suppression = head is actively pushing *against* copying that token.

In [ ]:
result = copy_suppression_score(model, tokens, layer=0, head=0)

print(f"Suppression scores per source position: {result['suppression_scores']}")
print(f"Mean suppression: {result['mean_suppression']:.4f}")
print(f"Max suppression at position: {result['max_suppression_pos']}")
print(f"Is copy-suppressing: {result['is_copy_suppressing']}")

## 2. Find Negative Heads

Scan all heads to identify which ones have copy-suppressing OV circuits. These are the "negative name mover" heads in the IOI circuit.

In [ ]:
neg = find_negative_heads(model, tokens, top_k=5)

print(f"Number of negative heads: {neg['n_negative_heads']}")
print(f"\nTop negative heads:")
for layer, head, score in neg['top_negative_heads']:
    print(f"  L{layer}H{head}: score={score:.4f}")

## 3. Suppression Per Attended Token

For a specific head, see the attention-weighted suppression at each source position. Shows exactly which tokens are being suppressed and by how much.

In [ ]:
supp = suppression_per_attended_token(model, tokens, layer=0, head=0)

print(f"Token suppression: {supp['token_suppression']}")
print(f"Attention weights: {[f'{w:.3f}' for w in supp['attention_weights']]}")
print(f"Most suppressed position: {supp['most_suppressed_pos']}")
print(f"Total suppression: {supp['total_suppression']:.4f}")

## 4. Copy vs Suppress Decomposition

Decompose all heads into those that promote (copy) vs suppress a target token. Reveals the balance of forces shaping the prediction.

In [ ]:
decomp = copy_vs_suppress_decomposition(model, tokens)

print(f"Head contributions shape: {decomp['head_contributions'].shape}")
print(f"Copy heads: {decomp['copy_heads']}")
print(f"Suppress heads: {decomp['suppress_heads']}")
print(f"Net effect: {decomp['net_effect']:.4f}")

## 5. Suppression Circuit on IOI

Analyze copy suppression on IOI-style prompts: identify which heads suppress the subject name at the position where the indirect object should be predicted.

In [ ]:
# IOI-style: subject at pos 1, indirect object at pos 3
ioi = suppression_circuit_on_ioi(model, tokens, subject_pos=1, io_pos=3)

print(f"Subject token: {ioi['subject_token']}, IO token: {ioi['io_token']}")
print(f"\nTop subject suppressors:")
for layer, head, score in ioi['top_suppressors']:
    print(f"  L{layer}H{head}: {score:.4f}")
print(f"\nNet effect by head (positive = good for IOI):")
print(ioi['net_effect_by_head'])